
# Eadro RCA — publication-safe experiment notebook

Run this notebook top-to-bottom in Google Colab.

It:
- installs the Eadro Python 3.8 environment;
- downloads and preprocesses SocialNetwork (SN) and TrainTicket (TT);
- makes the original train/test split reproducible;
- creates a validation split only from the original training set;
- selects checkpoints using validation HR@1;
- touches the test set only once per run;
- logs every epoch;
- runs 3 seeds for each configuration;
- compares Original Eadro, LSTM+Attention (α=0.5/0.6/0.7), and Hybrid DCC+LSTM (α=0.5);
- saves histories, checkpoints, raw results, mean±std tables, plots, and a real-sample walkthrough.

The full SN+TT grid is computationally substantial. Completed runs are resumable.


## 1. Environment and repository

In [ ]:

%cd /content

!test -x /content/bin/micromamba || (wget -qO- https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba)
!test -x /content/eadro-env/bin/python || /content/bin/micromamba create -y -p /content/eadro-env python=3.8 pip

!/content/eadro-env/bin/pip install -q \
    numpy==1.21.5 \
    pandas==1.4.2 \
    scikit-learn==1.1.2 \
    torch==1.12.1 \
    dgl==0.9.1 \
    tick==0.7.0.1 \
    drain3 \
    matplotlib

!test -d /content/Eadro-full || git clone https://github.com/FeiGSSS/Eadro.git /content/Eadro-full

!/content/eadro-env/bin/python -c "import torch,dgl,numpy,pandas,sklearn,tick,drain3; print('Torch',torch.__version__,'| DGL',dgl.__version__,'| environment OK')"


/content
bin/micromamba
[+] 0.0s
[+] 0.1s
Fetch Shard Index for conda-forge/linux-64                                                ✔ Done (0.1 sec)
Fetch Shard Index for conda-forge/noarch                                                  ✔ Done (0.2 sec)
Fetching and Parsing Packages' Shards                                                     ✔ Done (9.2 sec)

Resolving Environment                                                                     ✔ Done (1.4 sec)

Transaction

  Prefix: /content/eadro-env

  Updating specs:

   - python=3.8
   - pip


  Package               Version  Build                 Channel          Size
──────────────────────────────────────────────────────────────────────────────
  Install:
──────────────────────────────────────────────────────────────────────────────

  + _openmp_mutex           4.5  20_gnu                conda-forge      29kB
  + bzip2                 1.0.8  hda65f42_10           conda-forge     258kB
  + ca-certificates   2026.7.22  hbd8a

## 2. Download SN and TT datasets

In [ ]:
%cd /content
!mkdir -p /content/eadro-data
%cd /content/eadro-data

# SN already downloaded, preprocessed and trained in a separate notebook
# (Eadro_SN_only_publication_safe) -- those results are final and verified,
# so this notebook only needs to pull TT.
!test -f "TT Dataset.zip" || wget -q --show-progress -O "TT Dataset.zip" "https://zenodo.org/records/7615394/files/TT%20Dataset.zip?download=1"
!test -d "TT Dataset" || unzip -q "TT Dataset.zip"

!find /content/eadro-data -maxdepth 2 -type d | sort | head -30


## 3. Preprocess and build deterministic original train/test splits

In [ ]:
%env ROOT_PATH=/content/eadro-data
%cd /content/Eadro-full/codes/preprocess

# Fork compatibility fix.
!sed -i 's/groupby("event")/groupby("events")/' /content/Eadro-full/codes/preprocess/single_process.py

# TT-only fix: these four scripts default to looping over ['SN', 'TT'].
# SN is already done (separate notebook); restrict to TT only here so we
# don't waste time reprocessing SN raw data we won't retrain.
!sed -i "s/for dataset_name in \['SN', 'TT'\]:/for dataset_name in ['TT']:/" parse_raw_records.py
!sed -i "s/for dataset_name in \['SN', 'TT'\]:/for dataset_name in ['TT']:/" parse_raw_metrics.py
!sed -i "s/for dataset_name in \['SN', 'TT'\]:/for dataset_name in ['TT']:/" parse_raw_traces.py
!sed -i "s/for dataset_name in \['SN', 'TT'\]:/for dataset_name in ['TT']:/" parse_raw_logs.py

!grep -n "for dataset_name in" parse_raw_records.py parse_raw_metrics.py parse_raw_traces.py parse_raw_logs.py

# Deterministic preprocessing split.
!grep -q "EADRO_REPRO_SEED" align.py || sed -i '/import random/a random.seed(2026)  # EADRO_REPRO_SEED' align.py
!grep -q "EADRO_NUMPY_REPRO_SEED" align.py || sed -i '/import numpy as np/a np.random.seed(2026)  # EADRO_NUMPY_REPRO_SEED' align.py

# Raw -> parsed_data (TT only). Skip expensive stages when outputs already exist.
!test -f parsed_data/TT/records0.json || /content/eadro-env/bin/python parse_raw_records.py
!test -d parsed_data/TT/metrics0 || /content/eadro-env/bin/python parse_raw_metrics.py
!test -f parsed_data/TT/traces0.json || /content/eadro-env/bin/python parse_raw_traces.py
!test -f parsed_data/TT/templates.json || (/content/eadro-env/bin/python parse_raw_logs.py > /content/parse_raw_logs_output.txt 2>&1)

# Rebuild only the final split file for TT using the deterministic seed above.
!rm -f /content/Eadro-full/codes/chunks/TT/chunk_train.pkl /content/Eadro-full/codes/chunks/TT/chunk_test.pkl

!/content/eadro-env/bin/python align.py --name TT --chunk_lenth 10 --test_ratio 0.3

!ls -lh /content/Eadro-full/codes/chunks/TT



## 4. Publication-safe experiment protocol

The original `base.py` selects the best checkpoint by evaluating the test set during training. This notebook deliberately avoids `BaseModel.fit()`.

Protocol:

```text
original chunk_train.pkl
        |
        +---- 80% train
        |
        +---- 20% validation  -> choose best checkpoint by validation HR@1

original chunk_test.pkl       -> evaluate once, after training is finished
```

The train/validation split is fixed with split seed `2026` and is reused for every architecture and model seed.

The evaluator below mirrors the repository's metric implementation so comparisons remain directly compatible with the Eadro code.


In [ ]:

%%bash
set -e

cd /content/Eadro-full/codes

/content/eadro-env/bin/python <<'PY'
import os
import sys
import json
import time
import copy
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import dgl

from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import ndcg_score

sys.path.insert(0, "/content/Eadro-full/codes")

import model as eadro_model
from utils import read_json, load_chunks, seed_everything

# ============================================================
# EXPERIMENT GRID
# ============================================================

DATASETS = ["TT"]  # SN already trained & verified in Eadro_SN_only_publication_safe
MODEL_SEEDS = [42, 123, 2026]

EPOCHS = 50
BATCH_SIZE = 256
LEARNING_RATE = 0.001

VAL_RATIO_FROM_ORIGINAL_TRAIN = 0.20
SPLIT_SEED = 2026
SELECT_METRIC = "HR@1"

# TRIMMED FOR TT: only the 2 configs needed for the paper's core claim.
# The full alpha ablation (0.5/0.6/0.7) and the Hybrid variant were already
# done on SN -- LSTM alpha=0.5 came out as the best localization config
# there (highest HR@1/HR@3/NDCG@3), so this validates that SAME config
# generalizes to a second, independent dataset (TT) against the Baseline
# anchor, instead of re-running the whole 5-config sweep a second time.
# This cuts TT from 15 runs (5 configs x 3 seeds) down to 6 (2 configs x 3
# seeds) -- still 3 seeds, so you keep a real mean +/- std for TT too.
#
# If even 6 runs is too slow/disconnect-prone, cut MODEL_SEEDS below to a
# single seed (e.g. MODEL_SEEDS = [42]) to get it down to just 2 runs total
# -- note the paper then says TT used 1 seed vs. SN's 3, which is an honest
# caveat, not a problem.
CONFIGS = [
    {"name": "Baseline",     "encoder": "baseline", "alpha": 0.5},
    {"name": "LSTM_a0.5",    "encoder": "lstm",     "alpha": 0.5},
]

RESUME_COMPLETED_RUNS = True

RESULT_ROOT = Path("/content/eadro-publication-results")
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

# ============================================================
# RESUME FROM DRIVE (survive Colab disconnects)
# If a previous run of this notebook backed its results up to Drive,
# pull them back before training so already-finished runs are skipped
# instead of retrained from scratch.
# ============================================================
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    _drive_dir = Path("/content/drive/MyDrive/eadro-paper")
    _backup_zip = _drive_dir / "eadro_tt_results.zip"
    if _backup_zip.exists():
        shutil.unpack_archive(str(_backup_zip), str(RESULT_ROOT))
        print(f"[resume] restored previous TT results from {_backup_zip}", flush=True)
    else:
        print("[resume] no previous Drive backup found -- starting fresh.", flush=True)
except Exception as e:
    print(f"[resume] Drive restore skipped ({e})", flush=True)


def backup_to_drive(reason=""):
    """Zip RESULT_ROOT and push it to Drive. Called after every single run
    so a disconnect loses at most one run's progress, not the whole grid."""
    try:
        if not Path("/content/drive/MyDrive").exists():
            return
        drive_dir = Path("/content/drive/MyDrive/eadro-paper")
        drive_dir.mkdir(parents=True, exist_ok=True)
        archive_base = "/content/eadro_tt_results_backup"
        shutil.make_archive(archive_base, "zip", root_dir=str(RESULT_ROOT))
        shutil.copy(archive_base + ".zip", drive_dir / "eadro_tt_results.zip")
        print(f"[backup] saved to Drive {reason}", flush=True)
    except Exception as e:
        print(f"[backup] failed {reason}: {e}", flush=True)


INPROGRESS_DIR_NAME = "tt_inprogress"


def _drive_inprogress_dir():
    d = Path("/content/drive/MyDrive/eadro-paper") / INPROGRESS_DIR_NAME
    d.mkdir(parents=True, exist_ok=True)
    return d


def pull_inprogress_checkpoint(run_name, ckpt_file):
    """A run that was interrupted mid-training on a now-dead VM: pull its last
    saved epoch checkpoint back from Drive so training resumes from there
    instead of restarting the whole run from epoch 1."""
    try:
        if not Path("/content/drive/MyDrive").exists():
            return
        src = _drive_inprogress_dir() / f"{run_name}.pt"
        if src.exists() and not ckpt_file.exists():
            shutil.copy(src, ckpt_file)
            print(f"[resume] restored in-progress checkpoint for {run_name}", flush=True)
    except Exception as e:
        print(f"[resume] checkpoint restore skipped for {run_name} ({e})", flush=True)


def push_inprogress_checkpoint(run_name, ckpt_file):
    """Called after every epoch: sync just this run's checkpoint to Drive.
    Cheap (one small file) -- unlike backup_to_drive(), which zips every
    completed run and is only called once a run fully finishes."""
    try:
        if not Path("/content/drive/MyDrive").exists():
            return
        shutil.copy(ckpt_file, _drive_inprogress_dir() / f"{run_name}.pt")
    except Exception as e:
        print(f"[resume] checkpoint sync failed for {run_name} ({e})", flush=True)


def clear_inprogress_checkpoint(run_name, ckpt_file):
    try:
        if ckpt_file.exists():
            ckpt_file.unlink()
        if Path("/content/drive/MyDrive").exists():
            drive_ckpt = _drive_inprogress_dir() / f"{run_name}.pt"
            if drive_ckpt.exists():
                drive_ckpt.unlink()
    except Exception as e:
        print(f"[resume] checkpoint cleanup failed for {run_name} ({e})", flush=True)

# ============================================================
# MODEL VARIANTS
# ============================================================

OriginalMetricModel = eadro_model.MetricModel


class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )
        self.last_weights = None

    def forward(self, hidden_states):
        scores = self.score(hidden_states).squeeze(-1)
        weights = torch.softmax(scores, dim=1)
        self.last_weights = weights.detach().cpu()
        return torch.sum(hidden_states * weights.unsqueeze(-1), dim=1)


class LSTMMetricModel(nn.Module):
    def __init__(self, metric_num, device="cpu", metric_hiddens=[64], **kwargs):
        super().__init__()
        self.metric_num = metric_num
        hidden_dim = metric_hiddens[-1]

        self.lstm = nn.LSTM(
            input_size=metric_num,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True
        )
        self.attention = TemporalAttention(hidden_dim)
        self.out_dim = hidden_dim
        self.to(device)

    def forward(self, x):
        assert x.shape[-1] == self.metric_num
        hidden_states, _ = self.lstm(x.float())
        return self.attention(hidden_states)


class HybridMetricModel(nn.Module):
    def __init__(
        self,
        metric_num,
        device="cpu",
        metric_hiddens=[64],
        metric_kernel_sizes=[2],
        self_attn=True,
        chunk_lenth=10,
        **kwargs
    ):
        super().__init__()

        hidden_dim = metric_hiddens[-1]
        self.self_attn = self_attn

        self.dcc = eadro_model.ConvNet(
            metric_num,
            num_channels=metric_hiddens,
            kernel_sizes=metric_kernel_sizes,
            dev=device
        )
        self.dcc_attention = eadro_model.SelfAttention(hidden_dim, chunk_lenth)

        self.lstm = nn.LSTM(
            input_size=metric_num,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True
        )
        self.lstm_attention = TemporalAttention(hidden_dim)

        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU()
        )
        self.out_dim = hidden_dim
        self.to(device)

    def forward(self, x):
        x = x.float()

        dcc_hidden = self.dcc(x)
        if self.self_attn:
            dcc_embedding = self.dcc_attention(dcc_hidden)
        else:
            dcc_embedding = dcc_hidden[:, -1, :]

        lstm_hidden, _ = self.lstm(x)
        lstm_embedding = self.lstm_attention(lstm_hidden)

        return self.fusion(torch.cat([dcc_embedding, lstm_embedding], dim=-1))


def select_metric_encoder(kind):
    if kind == "baseline":
        eadro_model.MetricModel = OriginalMetricModel
    elif kind == "lstm":
        eadro_model.MetricModel = LSTMMetricModel
    elif kind == "hybrid":
        eadro_model.MetricModel = HybridMetricModel
    else:
        raise ValueError(kind)


# ============================================================
# DATASET
# ============================================================

class ChunkDataset(Dataset):
    def __init__(self, chunks, node_num, edges):
        self.data = []

        for chunk_id, chunk in chunks.items():
            graph = dgl.graph((edges[0], edges[1]), num_nodes=node_num)
            graph.ndata["logs"] = torch.FloatTensor(chunk["logs"])
            graph.ndata["metrics"] = torch.FloatTensor(chunk["metrics"])
            graph.ndata["traces"] = torch.FloatTensor(chunk["traces"])

            self.data.append((graph, int(chunk["culprit"])))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


def collate(batch):
    graphs, labels = map(list, zip(*batch))
    return dgl.batch(graphs), torch.tensor(labels, dtype=torch.long)


def make_train_val_split(train_chunks, val_ratio, split_seed):
    ids = list(train_chunks.keys())
    culprit = np.array([int(train_chunks[k]["culprit"]) for k in ids])

    try:
        train_ids, val_ids = train_test_split(
            ids,
            test_size=val_ratio,
            random_state=split_seed,
            stratify=culprit
        )
    except ValueError:
        anomaly = (culprit != -1).astype(int)
        train_ids, val_ids = train_test_split(
            ids,
            test_size=val_ratio,
            random_state=split_seed,
            stratify=anomaly
        )

    train_dict = {k: train_chunks[k] for k in train_ids}
    val_dict = {k: train_chunks[k] for k in val_ids}
    return train_dict, val_dict


# ============================================================
# REPOSITORY-COMPATIBLE EVALUATION
# ============================================================

def evaluate(model, loader, device):
    model.eval()

    hrs = np.zeros(5)
    ndcgs = np.zeros(5)
    TP = FP = FN = 0
    losses = []

    with torch.no_grad():
        for graph, ground_truths in loader:
            res = model(graph.to(device), ground_truths)
            losses.append(float(res["loss"].item()))

            for idx, faulty_nodes in enumerate(res["y_pred"]):
                culprit = ground_truths[idx].item()

                if culprit == -1:
                    if faulty_nodes[0] == -1:
                        TP += 1
                    else:
                        FP += 1
                else:
                    if faulty_nodes[0] == -1:
                        FN += 1
                    else:
                        TP += 1
                        rank = list(faulty_nodes).index(culprit)

                        for j in range(5):
                            hrs[j] += int(rank <= j)
                            ndcgs[j] += ndcg_score(
                                [res["y_prob"][idx]],
                                [res["pred_prob"][idx]],
                                k=j + 1
                            )

    pos = TP + FN

    results = {
        "loss": float(np.mean(losses)) if losses else np.nan,
        "F1": TP * 2.0 / (TP + FP + pos) if (TP + FP + pos) > 0 else 0.0,
        "Rec": TP / pos if pos > 0 else 0.0,
        "Pre": TP / (TP + FP) if (TP + FP) > 0 else 0.0,
    }

    for j in [1, 3, 5]:
        results[f"HR@{j}"] = hrs[j - 1] / pos if pos > 0 else 0.0
        results[f"ndcg@{j}"] = ndcgs[j - 1] / pos if pos > 0 else 0.0

    return results


# ============================================================
# SAFE TRAINING: VAL SELECTS CHECKPOINT, TEST ONCE
# ============================================================

def run_one(dataset_name, config, model_seed):
    data_dir = Path("./chunks") / dataset_name
    metadata = read_json(str(data_dir / "metadata.json"))

    event_num = metadata["event_num"]
    node_num = metadata["node_num"]
    metric_num = metadata["metric_num"]
    edges = metadata["edges"]
    chunk_lenth = metadata["chunk_lenth"]

    original_train, final_test = load_chunks(str(data_dir))

    train_chunks, val_chunks = make_train_val_split(
        original_train,
        VAL_RATIO_FROM_ORIGINAL_TRAIN,
        SPLIT_SEED
    )

    run_name = f"{dataset_name}__{config['name']}__seed{model_seed}"
    run_dir = RESULT_ROOT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    summary_file = run_dir / "summary.json"

    if RESUME_COMPLETED_RUNS and summary_file.exists():
        with open(summary_file, "r") as f:
            saved = json.load(f)
        print(f"SKIP completed: {run_name}", flush=True)
        return saved

    ckpt_file = run_dir / "checkpoint.pt"
    pull_inprogress_checkpoint(run_name, ckpt_file)

    with open(run_dir / "split.json", "w") as f:
        json.dump(
            {
                "dataset": dataset_name,
                "original_train_n": len(original_train),
                "train_n": len(train_chunks),
                "val_n": len(val_chunks),
                "test_n": len(final_test),
                "split_seed": SPLIT_SEED,
                "val_ratio_from_original_train": VAL_RATIO_FROM_ORIGINAL_TRAIN,
            },
            f,
            indent=2
        )

    train_ds = ChunkDataset(train_chunks, node_num, edges)
    val_ds = ChunkDataset(val_chunks, node_num, edges)
    test_ds = ChunkDataset(final_test, node_num, edges)

    generator = torch.Generator()
    generator.manual_seed(model_seed)

    train_dl = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate,
        generator=generator
    )
    val_dl = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate
    )
    test_dl = DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate
    )

    seed_everything(model_seed)
    random.seed(model_seed)
    np.random.seed(model_seed)
    torch.manual_seed(model_seed)

    device = torch.device("cpu")

    select_metric_encoder(config["encoder"])

    params = {
        "self_attn": True,
        "fuse_dim": 128,
        "alpha": config["alpha"],
        "locate_hiddens": [64],
        "detect_hiddens": [64],
        "log_dim": 16,
        "trace_kernel_sizes": [2],
        "trace_hiddens": [64],
        "metric_kernel_sizes": [2],
        "metric_hiddens": [64],
        "graph_hiddens": [64],
        "attn_head": 4,
        "activation": 0.2,
        "chunk_lenth": chunk_lenth,
    }

    model = eadro_model.MainModel(
        event_num,
        metric_num,
        node_num,
        device,
        **params
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    best_value = -float("inf")
    best_epoch = None
    best_state = None
    history = []
    start_time = time.time()
    start_epoch = 1

    if ckpt_file.exists():
        try:
            ckpt = torch.load(ckpt_file, map_location=device)
            model.load_state_dict(ckpt["model_state"])
            optimizer.load_state_dict(ckpt["optimizer_state"])
            best_value = ckpt["best_value"]
            best_epoch = ckpt["best_epoch"]
            best_state = ckpt["best_state"]
            history = ckpt["history"]
            start_epoch = ckpt["epoch"] + 1
            print(
                f"[resume] {run_name}: continuing from epoch {start_epoch}/{EPOCHS} "
                f"(best_val_HR1={best_value:.4f}@{best_epoch})",
                flush=True
            )
        except Exception as e:
            print(f"[resume] failed to load checkpoint for {run_name}, starting fresh ({e})", flush=True)
            start_epoch = 1

    print(
        f"\nSTART {run_name} | "
        f"train={len(train_ds)} val={len(val_ds)} test={len(test_ds)} "
        f"| resuming_from_epoch={start_epoch}",
        flush=True
    )

    for epoch in range(start_epoch, EPOCHS + 1):
        model.train()
        train_losses = []
        epoch_start = time.time()

        for graph, labels in train_dl:
            optimizer.zero_grad()
            res = model(graph.to(device), labels)
            loss = res["loss"]
            loss.backward()
            optimizer.step()
            train_losses.append(float(loss.item()))

        train_loss = float(np.mean(train_losses))
        val_results = evaluate(model, val_dl, device)

        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                **{f"val_{k}": v for k, v in val_results.items()}
            }
        )

        current = val_results[SELECT_METRIC]

        if current > best_value:
            best_value = current
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            torch.save(
                best_state,
                run_dir / "best_model.pt",
                _use_new_zipfile_serialization=False
            )

        print(
            f"[{dataset_name}][{config['name']}][seed={model_seed}] "
            f"epoch {epoch:02d}/{EPOCHS} "
            f"loss={train_loss:.5f} "
            f"val_F1={val_results['F1']:.4f} "
            f"val_HR1={val_results['HR@1']:.4f} "
            f"val_HR3={val_results['HR@3']:.4f} "
            f"best_val_HR1={best_value:.4f}@{best_epoch} "
            f"[{time.time()-epoch_start:.1f}s]",
            flush=True
        )

        torch.save(
            {
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "best_value": best_value,
                "best_epoch": best_epoch,
                "best_state": best_state,
                "history": history,
            },
            ckpt_file,
            _use_new_zipfile_serialization=False,
        )
        push_inprogress_checkpoint(run_name, ckpt_file)

    pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)

    model.load_state_dict(best_state)

    # The final test loader is used here, once, after model selection is complete.
    test_results = evaluate(model, test_dl, device)

    summary = {
        "dataset": dataset_name,
        "config": config["name"],
        "encoder": config["encoder"],
        "alpha": config["alpha"],
        "seed": model_seed,
        "best_epoch": best_epoch,
        "best_val_HR@1": best_value,
        "runtime_seconds": time.time() - start_time,
        **{f"test_{k}": v for k, v in test_results.items()},
    }

    with open(summary_file, "w") as f:
        json.dump(summary, f, indent=2)

    clear_inprogress_checkpoint(run_name, ckpt_file)

    print(
        "TEST ONCE -> "
        f"F1={test_results['F1']:.4f}, "
        f"Pre={test_results['Pre']:.4f}, "
        f"Rec={test_results['Rec']:.4f}, "
        f"HR@1={test_results['HR@1']:.4f}, "
        f"HR@3={test_results['HR@3']:.4f}, "
        f"NDCG@3={test_results['ndcg@3']:.4f}",
        flush=True
    )

    return summary


all_summaries = []

for dataset_name in DATASETS:
    for config in CONFIGS:
        for model_seed in MODEL_SEEDS:
            all_summaries.append(run_one(dataset_name, config, model_seed))
            backup_to_drive(reason=f"(after {dataset_name}__{config['name']}__seed{model_seed})")

raw_df = pd.DataFrame(all_summaries)
raw_df.to_csv(RESULT_ROOT / "summary_runs.csv", index=False)

print("\n================ RAW TEST RESULTS ================")
print(
    raw_df[
        [
            "dataset", "config", "seed", "best_epoch",
            "test_F1", "test_Pre", "test_Rec",
            "test_HR@1", "test_HR@3", "test_ndcg@3"
        ]
    ].to_string(index=False)
)

print("\n================ MEAN ± STD ================")

for dataset_name in DATASETS:
    print(f"\n{dataset_name}")
    subset = raw_df[raw_df["dataset"] == dataset_name]

    for config_name in [c["name"] for c in CONFIGS]:
        s = subset[subset["config"] == config_name]

        def ms(col):
            return f"{s[col].mean():.4f} ± {s[col].std(ddof=1):.4f}"

        print(
            f"{config_name:14s} | "
            f"F1 {ms('test_F1')} | "
            f"HR@1 {ms('test_HR@1')} | "
            f"HR@3 {ms('test_HR@3')} | "
            f"NDCG@3 {ms('test_ndcg@3')}"
        )

print(f"\nSaved to: {RESULT_ROOT}")
PY


## 5. Inspect final seeded test results

In [ ]:

import pandas as pd
from pathlib import Path

RESULT_ROOT = Path("/content/eadro-publication-results")
runs = pd.read_csv(RESULT_ROOT / "summary_runs.csv")

runs[
    [
        "dataset", "config", "seed", "best_epoch",
        "test_F1", "test_Pre", "test_Rec",
        "test_HR@1", "test_HR@3", "test_HR@5",
        "test_ndcg@3", "test_ndcg@5"
    ]
]


In [ ]:

metrics = [
    "test_F1", "test_Pre", "test_Rec",
    "test_HR@1", "test_HR@3", "test_HR@5",
    "test_ndcg@3", "test_ndcg@5"
]

rows = []

for (dataset, config), g in runs.groupby(["dataset", "config"], sort=False):
    row = {"Dataset": dataset, "Model": config}

    for metric in metrics:
        row[metric.replace("test_", "")] = (
            f"{g[metric].mean():.4f} ± {g[metric].std(ddof=1):.4f}"
        )

    rows.append(row)

paper_table = pd.DataFrame(rows)
paper_table


## 6. Graphs

In [ ]:

import matplotlib.pyplot as plt

means = (
    runs.groupby(["dataset", "config"], as_index=False)
        [["test_F1", "test_HR@1"]]
        .mean()
)

for dataset in means["dataset"].unique():
    d = means[means["dataset"] == dataset]

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(d["test_F1"], d["test_HR@1"], s=90)

    for _, row in d.iterrows():
        ax.annotate(
            row["config"],
            (row["test_F1"], row["test_HR@1"]),
            xytext=(5, 5),
            textcoords="offset points"
        )

    ax.set_xlabel("Anomaly-detection F1")
    ax.set_ylabel("Root-cause localization HR@1")
    ax.set_title(f"{dataset}: Detection–Localization Trade-off")
    ax.grid(alpha=0.25)
    plt.show()


In [ ]:

metric_map = {
    "HR@1": "test_HR@1",
    "HR@3": "test_HR@3",
    "NDCG@3": "test_ndcg@3"
}

for dataset in runs["dataset"].unique():
    d = runs[runs["dataset"] == dataset]
    configs = list(dict.fromkeys(d["config"].tolist()))
    x = range(len(configs))

    fig, ax = plt.subplots(figsize=(11, 6))
    width = 0.24
    offsets = [-width, 0, width]

    for offset, (label, col) in zip(offsets, metric_map.items()):
        means_plot = [d[d["config"] == c][col].mean() for c in configs]
        stds_plot = [d[d["config"] == c][col].std(ddof=1) for c in configs]

        ax.bar(
            [i + offset for i in x],
            means_plot,
            width=width,
            yerr=stds_plot,
            capsize=3,
            label=label
        )

    ax.set_xticks(list(x))
    ax.set_xticklabels(configs, rotation=20)
    ax.set_ylabel("Score")
    ax.set_title(f"{dataset}: RCA performance (mean ± std)")
    ax.legend()
    ax.grid(axis="y", alpha=0.2)
    plt.tight_layout()
    plt.show()


In [ ]:

metric_map = {
    "F1": "test_F1",
    "Precision": "test_Pre",
    "Recall": "test_Rec"
}

for dataset in runs["dataset"].unique():
    d = runs[runs["dataset"] == dataset]
    configs = list(dict.fromkeys(d["config"].tolist()))
    x = range(len(configs))

    fig, ax = plt.subplots(figsize=(11, 6))
    width = 0.24
    offsets = [-width, 0, width]

    for offset, (label, col) in zip(offsets, metric_map.items()):
        means_plot = [d[d["config"] == c][col].mean() for c in configs]
        stds_plot = [d[d["config"] == c][col].std(ddof=1) for c in configs]

        ax.bar(
            [i + offset for i in x],
            means_plot,
            width=width,
            yerr=stds_plot,
            capsize=3,
            label=label
        )

    ax.set_xticks(list(x))
    ax.set_xticklabels(configs, rotation=20)
    ax.set_ylabel("Score")
    ax.set_title(f"{dataset}: anomaly-detection performance (mean ± std)")
    ax.legend()
    ax.grid(axis="y", alpha=0.2)
    plt.tight_layout()
    plt.show()


In [ ]:

for dataset in runs["dataset"].unique():
    fig, ax = plt.subplots(figsize=(11, 6))

    for config in runs["config"].unique():
        history_file = RESULT_ROOT / f"{dataset}__{config}__seed42" / "history.csv"

        if history_file.exists():
            h = pd.read_csv(history_file)
            ax.plot(h["epoch"], h["val_HR@1"], label=config)

    ax.set_xlabel("Epoch")
    ax.set_ylabel("Validation HR@1")
    ax.set_title(f"{dataset}: validation HR@1 trajectory (seed 42)")
    ax.legend()
    ax.grid(alpha=0.25)
    plt.show()

for dataset in runs["dataset"].unique():
    fig, ax = plt.subplots(figsize=(11, 6))

    for config in runs["config"].unique():
        history_file = RESULT_ROOT / f"{dataset}__{config}__seed42" / "history.csv"

        if history_file.exists():
            h = pd.read_csv(history_file)
            ax.plot(h["epoch"], h["train_loss"], label=config)

    ax.set_xlabel("Epoch")
    ax.set_ylabel("Training loss")
    ax.set_title(f"{dataset}: training-loss trajectory (seed 42)")
    ax.legend()
    ax.grid(alpha=0.25)
    plt.show()


## 7. Real faulty-sample walkthrough

In [ ]:

import pickle
import json
import numpy as np
import matplotlib.pyplot as plt

DATASET = "SN"  # change to "TT" for a TrainTicket sample

test_path = f"/content/Eadro-full/codes/chunks/{DATASET}/chunk_test.pkl"
metadata_path = f"/content/Eadro-full/codes/chunks/{DATASET}/metadata.json"

with open(test_path, "rb") as f:
    test_chunks = pickle.load(f)

with open(metadata_path, "r") as f:
    metadata = json.load(f)

faulty_id = None
faulty_sample = None

for chunk_id, sample in test_chunks.items():
    if int(sample["culprit"]) != -1:
        faulty_id = chunk_id
        faulty_sample = sample
        break

if faulty_sample is None:
    raise RuntimeError("No faulty sample found.")

if DATASET == "SN":
    service_names = [
        "social-graph-service",
        "compose-post-service",
        "post-storage-service",
        "user-timeline-service",
        "url-shorten-service",
        "user-service",
        "media-service",
        "text-service",
        "unique-id-service",
        "user-mention-service",
        "home-timeline-service",
        "nginx-web-server",
    ]
else:
    service_names = [
        "ts-assurance-service", "ts-auth-service", "ts-basic-service",
        "ts-cancel-service", "ts-config-service", "ts-contacts-service",
        "ts-food-map-service", "ts-food-service", "ts-inside-payment-service",
        "ts-notification-service", "ts-order-other-service", "ts-order-service",
        "ts-payment-service", "ts-preserve-service", "ts-price-service",
        "ts-route-plan-service", "ts-route-service", "ts-seat-service",
        "ts-security-service", "ts-station-service", "ts-ticketinfo-service",
        "ts-train-service", "ts-travel-plan-service", "ts-travel-service",
        "ts-travel2-service", "ts-user-service", "ts-verification-code-service",
    ]

metric_names = [
    "cpu_usage_system",
    "cpu_usage_total",
    "cpu_usage_user",
    "memory_usage",
    "memory_working_set",
    "rx_bytes",
    "tx_bytes",
]

culprit = int(faulty_sample["culprit"])

print("Chunk ID:", faulty_id)
print("Ground-truth culprit ID:", culprit)
print("Ground-truth culprit service:", service_names[culprit])
print()
print("metrics shape:", np.asarray(faulty_sample["metrics"]).shape)
print("traces shape :", np.asarray(faulty_sample["traces"]).shape)
print("logs shape   :", np.asarray(faulty_sample["logs"]).shape)

print(r'''
Processing path for this real sample:

metrics [services, 10, 7] ---> KPI encoder -----------+
traces  [services, 10, 2] ---> trace encoder --------+--> gated multimodal fusion
logs    [services, events] ---> log encoder ----------+          |
                                                                v
                                                   service node features
                                                                |
                                                                v
                                                     dependency graph + GAT
                                                                |
                                                +---------------+---------------+
                                                |                               |
                                                v                               v
                                       anomaly-detection head          RCA ranking head
''')


In [ ]:

kpis = np.asarray(faulty_sample["metrics"])[culprit]
time_steps = np.arange(1, kpis.shape[0] + 1)

fig, ax = plt.subplots(figsize=(11, 6))

for i, metric_name in enumerate(metric_names):
    ax.plot(time_steps, kpis[:, i], marker="o", label=metric_name)

ax.set_xlabel("Window time step")
ax.set_ylabel("Normalized KPI value")
ax.set_title(
    f"{DATASET} faulty sample: KPI sequence for culprit {service_names[culprit]}"
)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:

fig, ax = plt.subplots(figsize=(10, 5))
image = ax.imshow(kpis.T, aspect="auto")

ax.set_xticks(range(kpis.shape[0]))
ax.set_xticklabels(range(1, kpis.shape[0] + 1))
ax.set_yticks(range(len(metric_names)))
ax.set_yticklabels(metric_names)

ax.set_xlabel("Window time step")
ax.set_title(f"{DATASET}: KPI heatmap for culprit {service_names[culprit]}")

plt.colorbar(image, ax=ax, label="Normalized KPI value")
plt.tight_layout()
plt.show()


In [ ]:

metrics_shape = tuple(np.asarray(faulty_sample["metrics"]).shape)
traces_shape = tuple(np.asarray(faulty_sample["traces"]).shape)
logs_shape = tuple(np.asarray(faulty_sample["logs"]).shape)

fig, ax = plt.subplots(figsize=(14, 5))
ax.axis("off")

boxes = [
    (0.02, 0.66, f"Metrics\n{metrics_shape}\nKPI encoder"),
    (0.02, 0.36, f"Traces\n{traces_shape}\nTrace encoder"),
    (0.02, 0.06, f"Logs\n{logs_shape}\nLog encoder"),
    (0.33, 0.36, "Gated multimodal\nfusion"),
    (0.56, 0.36, "Service dependency\ngraph + GAT"),
    (0.80, 0.55, "Anomaly head\nNormal / Abnormal"),
    (0.80, 0.17, f"RCA head\nRank services\nGT: {service_names[culprit]}"),
]

for x, y, text in boxes:
    ax.text(
        x, y, text,
        transform=ax.transAxes,
        ha="left",
        va="center",
        bbox=dict(boxstyle="round,pad=0.5", fill=False)
    )

arrows = [
    ((0.20, 0.66), (0.33, 0.44)),
    ((0.20, 0.36), (0.33, 0.40)),
    ((0.20, 0.06), (0.33, 0.35)),
    ((0.48, 0.40), (0.56, 0.40)),
    ((0.72, 0.40), (0.80, 0.58)),
    ((0.72, 0.40), (0.80, 0.22)),
]

for start, end in arrows:
    ax.annotate(
        "",
        xy=end,
        xytext=start,
        xycoords="axes fraction",
        textcoords="axes fraction",
        arrowprops=dict(arrowstyle="->")
    )

ax.set_title("One real Eadro sample: end-to-end processing")
plt.show()



## 8. Reporting rule

After this notebook is run, use the new **mean ± standard deviation** values as the paper results.

Do not mix the earlier single-run numbers with these results, because the experimental protocol is now different.

The paper comparisons are:

- **Baseline vs LSTM α=0.5**: effect of replacing the KPI temporal encoder.
- **LSTM α=0.5/0.6/0.7**: detection–localization loss-weight trade-off.
- **Baseline vs Hybrid α=0.5**: whether combining DCC and LSTM produces a better balanced representation.
- **SN vs TT**: whether the observed behavior generalizes across the two official Eadro datasets.



# 9. Paper-ready evidence package

Run this section after all experiments finish. It produces the numerical evidence needed for the paper:
- dataset/split statistics;
- seed-level results;
- mean ± standard deviation;
- baseline deltas;
- paired t-tests;
- Cohen's dz effect sizes;
- 95% confidence intervals;
- alpha trade-off tables;
- runtime/epoch summaries;
- model parameter counts;
- cross-dataset consistency tables;
- CSV exports and one ZIP archive.


In [ ]:

import json, math, pickle, shutil
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

RESULT_ROOT = Path("/content/eadro-publication-results")
runs = pd.read_csv(RESULT_ROOT / "summary_runs.csv")

CONFIG_ORDER = ["Baseline","LSTM_a0.5","LSTM_a0.6","LSTM_a0.7","Hybrid_a0.5"]
METRIC_COLS = ["test_F1","test_Pre","test_Rec","test_HR@1","test_HR@3","test_HR@5","test_ndcg@1","test_ndcg@3","test_ndcg@5"]

print("Completed runs:", len(runs))
display(runs.groupby(["dataset","config"]).size().rename("runs").reset_index())


## 9.1 Dataset and split statistics

In [ ]:

from sklearn.model_selection import train_test_split

def dataset_stats(dataset):
    base = Path(f"/content/Eadro-full/codes/chunks/{dataset}")
    with open(base/"metadata.json") as f: metadata = json.load(f)
    with open(base/"chunk_train.pkl","rb") as f: original_train = pickle.load(f)
    with open(base/"chunk_test.pkl","rb") as f: test = pickle.load(f)

    ids = list(original_train.keys())
    culprit = np.array([int(original_train[k]["culprit"]) for k in ids])
    try:
        tr_ids, va_ids = train_test_split(ids,test_size=0.20,random_state=2026,stratify=culprit)
    except ValueError:
        anomaly = (culprit != -1).astype(int)
        tr_ids, va_ids = train_test_split(ids,test_size=0.20,random_state=2026,stratify=anomaly)

    train = {k: original_train[k] for k in tr_ids}
    val = {k: original_train[k] for k in va_ids}

    def one(name,d):
        labels = np.array([int(v["culprit"]) for v in d.values()])
        faulty = labels != -1
        return {
            "Dataset":dataset,"Split":name,"Chunks":len(d),
            "Normal":int((~faulty).sum()),"Faulty":int(faulty.sum()),
            "Fault_rate":float(faulty.mean()),
            "Distinct_faulty_culprits":int(len(set(labels[faulty].tolist())))
        }

    sample = next(iter(original_train.values()))
    meta = {
        "Dataset":dataset,
        "Services":int(metadata["node_num"]),
        "KPI_features":int(metadata["metric_num"]),
        "Log_features":int(metadata["event_num"]),
        "Window_length":int(metadata["chunk_lenth"]),
        "Metric_tensor_shape":str(np.asarray(sample["metrics"]).shape),
        "Trace_tensor_shape":str(np.asarray(sample["traces"]).shape),
        "Log_tensor_shape":str(np.asarray(sample["logs"]).shape),
    }

    combined = {**original_train, **test}
    culprits = pd.DataFrame({
        "Dataset":dataset,
        "Culprit_ID":[int(v["culprit"]) for v in combined.values()]
    }).value_counts().reset_index(name="Chunk_count")

    return meta,[one("Train",train),one("Validation",val),one("Test",test)],culprits

metas,splits,culprits=[],[],[]
for ds in sorted(runs["dataset"].unique()):
    m,s,c=dataset_stats(ds)
    metas.append(m); splits.extend(s); culprits.append(c)

dataset_metadata=pd.DataFrame(metas)
split_statistics=pd.DataFrame(splits)
culprit_distribution=pd.concat(culprits,ignore_index=True)

display(dataset_metadata)
display(split_statistics)
display(culprit_distribution)

dataset_metadata.to_csv(RESULT_ROOT/"paper_dataset_metadata.csv",index=False)
split_statistics.to_csv(RESULT_ROOT/"paper_split_statistics.csv",index=False)
culprit_distribution.to_csv(RESULT_ROOT/"paper_culprit_distribution.csv",index=False)


## 9.2 Mean ± std and seed-level results

In [ ]:

rows=[]
for ds in sorted(runs["dataset"].unique()):
    for cfg in CONFIG_ORDER:
        g=runs[(runs.dataset==ds)&(runs.config==cfg)]
        if g.empty: continue
        row={"Dataset":ds,"Model":cfg,"Seeds":len(g)}
        for m in METRIC_COLS:
            short=m.replace("test_","")
            row[short]=f"{g[m].mean():.4f} ± {g[m].std(ddof=1):.4f}"
        rows.append(row)
paper_results=pd.DataFrame(rows)
display(paper_results)
paper_results.to_csv(RESULT_ROOT/"paper_main_results_mean_std.csv",index=False)

seed_level=runs[["dataset","config","seed","best_epoch","runtime_seconds"]+METRIC_COLS].sort_values(["dataset","config","seed"])
display(seed_level)
seed_level.to_csv(RESULT_ROOT/"paper_seed_level_results.csv",index=False)


## 9.3 Paired statistics against baseline

In [ ]:

def paired_comparison(dataset, config, metric):
    a=runs[(runs.dataset==dataset)&(runs.config=="Baseline")][["seed",metric]].rename(columns={metric:"baseline"})
    b=runs[(runs.dataset==dataset)&(runs.config==config)][["seed",metric]].rename(columns={metric:"model"})
    p=a.merge(b,on="seed").sort_values("seed")
    diff=(p.model-p.baseline).to_numpy()
    n=len(diff)
    md=float(np.mean(diff))
    sd=float(np.std(diff,ddof=1)) if n>1 else np.nan
    se=sd/math.sqrt(n) if n>1 else np.nan
    if n>1 and np.isfinite(se):
        tcrit=stats.t.ppf(0.975,df=n-1)
        lo,hi=md-tcrit*se,md+tcrit*se
        t,pv=stats.ttest_rel(p.model,p.baseline)
    else:
        lo=hi=t=pv=np.nan
    dz=md/sd if n>1 and sd>0 else np.nan
    return {
        "Dataset":dataset,"Comparison":f"{config} vs Baseline","Metric":metric.replace("test_",""),
        "n_seeds":n,"Baseline_mean":p.baseline.mean(),"Model_mean":p.model.mean(),
        "Mean_difference":md,"Difference_percentage_points":100*md,
        "CI95_low":lo,"CI95_high":hi,"paired_t":t,"p_value":pv,"Cohens_dz":dz,
        "Improved_in_all_seeds":bool(np.all(diff>0)),
        "Worse_in_all_seeds":bool(np.all(diff<0))
    }

rows=[]
for ds in sorted(runs.dataset.unique()):
    for cfg in ["LSTM_a0.5","LSTM_a0.6","LSTM_a0.7","Hybrid_a0.5"]:
        if not ((runs.dataset==ds)&(runs.config==cfg)).any(): continue
        for m in METRIC_COLS:
            rows.append(paired_comparison(ds,cfg,m))
paired_stats=pd.DataFrame(rows)

headline=paired_stats[paired_stats.Metric.isin(["F1","Pre","Rec","HR@1","HR@3","ndcg@3"])]
display(headline)

paired_stats.to_csv(RESULT_ROOT/"paper_paired_statistics_all_metrics.csv",index=False)
headline.to_csv(RESULT_ROOT/"paper_paired_statistics_headline.csv",index=False)


## 9.4 Alpha trade-off, runtime, and cross-dataset consistency

In [ ]:

alpha_map={"LSTM_a0.5":0.5,"LSTM_a0.6":0.6,"LSTM_a0.7":0.7}
rows=[]
for ds in sorted(runs.dataset.unique()):
    for cfg,a in alpha_map.items():
        g=runs[(runs.dataset==ds)&(runs.config==cfg)]
        if g.empty: continue
        rows.append({
            "Dataset":ds,"alpha":a,
            "F1_mean":g.test_F1.mean(),"F1_std":g.test_F1.std(ddof=1),
            "HR@1_mean":g["test_HR@1"].mean(),"HR@1_std":g["test_HR@1"].std(ddof=1),
            "HR@3_mean":g["test_HR@3"].mean(),"NDCG@3_mean":g["test_ndcg@3"].mean()
        })
alpha_tradeoff=pd.DataFrame(rows).sort_values(["Dataset","alpha"])
display(alpha_tradeoff)
alpha_tradeoff.to_csv(RESULT_ROOT/"paper_alpha_tradeoff.csv",index=False)

runtime_summary=(runs.groupby(["dataset","config"]).agg(
    runtime_mean_seconds=("runtime_seconds","mean"),
    runtime_std_seconds=("runtime_seconds","std"),
    best_epoch_mean=("best_epoch","mean"),
    best_epoch_std=("best_epoch","std"),
    best_val_HR1_mean=("best_val_HR@1","mean"),
    best_val_HR1_std=("best_val_HR@1","std")
).reset_index())
display(runtime_summary)
runtime_summary.to_csv(RESULT_ROOT/"paper_runtime_and_epoch_summary.csv",index=False)

cross=[]
for cfg in ["LSTM_a0.5","LSTM_a0.6","LSTM_a0.7","Hybrid_a0.5"]:
    for m in ["test_F1","test_HR@1","test_HR@3","test_ndcg@3"]:
        row={"Configuration":cfg,"Metric":m.replace("test_","")}
        signs=[]
        for ds in sorted(runs.dataset.unique()):
            b=runs[(runs.dataset==ds)&(runs.config=="Baseline")][m].mean()
            x=runs[(runs.dataset==ds)&(runs.config==cfg)][m].mean()
            row[f"{ds}_delta"]=x-b
            signs.append(x-b>0)
        row["Improves_over_baseline_on_all_datasets"]=all(signs)
        cross.append(row)
cross_dataset=pd.DataFrame(cross)
display(cross_dataset)
cross_dataset.to_csv(RESULT_ROOT/"paper_cross_dataset_consistency.csv",index=False)


## 9.5 Claim-support table

In [ ]:

claim_rows=[]
for ds in sorted(runs.dataset.unique()):
    b=runs[(runs.dataset==ds)&(runs.config=="Baseline")]
    for cfg in ["LSTM_a0.5","LSTM_a0.6","LSTM_a0.7","Hybrid_a0.5"]:
        g=runs[(runs.dataset==ds)&(runs.config==cfg)]
        if g.empty: continue
        f1=paired_stats[(paired_stats.Dataset==ds)&(paired_stats.Comparison==f"{cfg} vs Baseline")&(paired_stats.Metric=="F1")].iloc[0]
        hr=paired_stats[(paired_stats.Dataset==ds)&(paired_stats.Comparison==f"{cfg} vs Baseline")&(paired_stats.Metric=="HR@1")].iloc[0]
        claim_rows.append({
            "Dataset":ds,"Configuration":cfg,
            "Baseline_F1":b.test_F1.mean(),"Model_F1":g.test_F1.mean(),
            "F1_delta_pp":100*(g.test_F1.mean()-b.test_F1.mean()),
            "F1_p":f1.p_value,"F1_Cohens_dz":f1.Cohens_dz,
            "Baseline_HR@1":b["test_HR@1"].mean(),"Model_HR@1":g["test_HR@1"].mean(),
            "HR@1_delta_pp":100*(g["test_HR@1"].mean()-b["test_HR@1"].mean()),
            "HR@1_p":hr.p_value,"HR@1_Cohens_dz":hr.Cohens_dz,
            "HR@1_improved_all_seeds":hr.Improved_in_all_seeds
        })
claim_support=pd.DataFrame(claim_rows)
display(claim_support)
claim_support.to_csv(RESULT_ROOT/"paper_claim_support.csv",index=False)


## 9.6 Paper-ready plots from final seeded results

In [ ]:

import matplotlib.pyplot as plt

for ds in alpha_tradeoff.Dataset.unique():
    d=alpha_tradeoff[alpha_tradeoff.Dataset==ds]
    fig,ax=plt.subplots(figsize=(7,5))
    ax.errorbar(d.alpha,d.F1_mean,yerr=d.F1_std,marker="o",capsize=4)
    ax.set_xlabel("α"); ax.set_ylabel("Test F1")
    ax.set_title(f"{ds}: anomaly-detection response to α")
    ax.grid(alpha=.25); plt.show()

    fig,ax=plt.subplots(figsize=(7,5))
    ax.errorbar(d.alpha,d["HR@1_mean"],yerr=d["HR@1_std"],marker="o",capsize=4)
    ax.set_xlabel("α"); ax.set_ylabel("Test HR@1")
    ax.set_title(f"{ds}: RCA response to α")
    ax.grid(alpha=.25); plt.show()



## 9.7 Claim guardrails

These experiments support claims about:
- replacing the full KPI encoder: **DCC + self-attention → LSTM + temporal attention**;
- the detection–localization trade-off across the tested α values 0.5, 0.6, 0.7;
- whether the hybrid encoder improves over baseline or tuned LSTM;
- whether trends reproduce across SN and TT;
- variability across the three tested seeds.

Do not claim that temporal attention alone caused the gain, because both the recurrent encoder and attention mechanism changed together.

Do not claim a continuously tunable frontier beyond the tested α values.

With only three seeds, report p-values together with effect sizes, confidence intervals, and seed-level results.


## 9.8 Export all paper evidence

In [ ]:

evidence_files=sorted(p.name for p in RESULT_ROOT.glob("paper_*.csv"))
manifest=pd.DataFrame({"Evidence_file":evidence_files})
display(manifest)
manifest.to_csv(RESULT_ROOT/"paper_evidence_manifest.csv",index=False)

archive_path=shutil.make_archive("/content/eadro_paper_evidence","zip",root_dir=RESULT_ROOT)
print("Evidence ZIP:",archive_path)


## 9.9 Optional Drive backup

In [ ]:

from google.colab import drive
drive.mount("/content/drive")
!mkdir -p "/content/drive/MyDrive/eadro-paper"
!cp -r /content/eadro-publication-results "/content/drive/MyDrive/eadro-paper/"
!cp /content/eadro_paper_evidence.zip "/content/drive/MyDrive/eadro-paper/"
print("Backed up to /content/drive/MyDrive/eadro-paper/")
